# Multimodal Short-Video Recommendation System

This project aims to build a recommendation system to improve content relevance on a mobile short-video platform. The system predicts user engagement with candidate videos, measured by watch ratio, which is then used to rank candidate videos and recommend the most relevant content to users.

In [1]:
# Import libraries
import pandas as pd

## Step 1: Data Loading

All datasets used in this project are sourced from the ShortVideo Dataset, obtained from the following repository: https://github.com/tsinghua-fib-lab/ShortVideo_dataset

1. **Interactions Dataset** (`interaction.csv`): Records large-scale interaction logs between users and short videos on a mobile short-video platform.

2. **Categories Dataset** (`categories_cn_en.csv`): Mapping between category IDs and their corresponding category names. Specifically, it maps `root_id`, `parent_id` and `category_id` to their respective category names, with labels available in both English and Chinese.

3. **Video Titles Dataset** (`/title_en` folder): Contains English titles of the short videos, stored as individual `.txt` files, each named according to their corresponding `pid`.

4. **ASR (Automatic Speech Recognition) Dataset** (`/asr_en` folder): Contains English transcriptions of the short video audio, stored as individual `.txt` files, each named according to their corresponding `pid`.

5. **Video Features Dataset** (`/video_feature_total` folder): Contains pre-extracted visual content embeddings for the short videos, stored as individual `.npy` files, each named according to their corresponding `pid`.

Datasets 3–5 are later merged with the video attributes to form `video_data_full.csv`, which consists of all video information.

### 1.1 Loading Raw Datasets

In [2]:
from google.colab import drive
drive.mount('/content/drive')

# read the CSV file (update the path to match your file's directory)
interaction_df = pd.read_csv('/content/drive/MyDrive/BT4222 Data/interaction.csv')
categories_df = pd.read_csv('/content/drive/MyDrive/BT4222 Data/categories_cn_en.csv')

Mounted at /content/drive


In [3]:
# Check the first few rows of the interaction dataset
interaction_df.head()

,user_id,pid,author_id,category_id,category_level,parent_id,root_id,exposed_time,author_fans_count,watch_time,...,tag_name,title,p_hour,p_date,gender,age,mod_price,fre_city,fre_community_type,fre_city_level
0,7313,84199269992,193159093,36,1,36,36,1663333100,46761,136,...,正能量,现如今为啥,21,20220916,M,42,2699,邯郸,unknown,三线城市
1,7313,84199269992,193159093,36,1,36,36,1663333100,46761,136,...,电视机,现如今为啥,21,20220916,M,42,2699,邯郸,unknown,三线城市
2,7313,84199269992,193159093,36,1,36,36,1663333100,46761,136,...,内容过于真实,现如今为啥,21,20220916,M,42,2699,邯郸,unknown,三线城市
3,7313,84199269992,193159093,373,2,36,36,1663333100,46761,136,...,正能量,现如今为啥,21,20220916,M,42,2699,邯郸,unknown,三线城市
4,7313,84199269992,193159093,373,2,36,36,1663333100,46761,136,...,电视机,现如今为啥,21,20220916,M,42,2699,邯郸,unknown,三线城市


In [4]:
# Shape of interaction dataset
print(f"Original Shape: {interaction_df.shape}")
print(f"Unique Interactions (user_id, pid, exposed_time): "
      f"{interaction_df.drop_duplicates(['user_id','pid','exposed_time']).shape[0]:,}")

Original Shape: (6767010, 28)
Unique Interactions (user_id, pid, exposed_time): 1,019,568


In [5]:
# Check the first few rows of the categories dataset
categories_df.head()

,category_level,category_id,category_name_cn,parent_id,root_id,category_name_en
0,2,134,美女,8,8,beauty
1,2,577,美食售卖,12,12,gourmet food sales
2,3,1256,产品与运营,163,14,Products and Operations
3,3,1248,熊猫,183,17,panda
4,2,204,理财,22,22,Financial management


In [6]:
# Shape of categories dataset
print(f"Original Shape: {categories_df.shape}")

Original Shape: (826, 6)


### 1.2 Normalising Interaction Dataset

#### 1.2.1 PID Label Encoding

In [7]:
# Sort unique PIDs ascending and assign sequential IDs 1 to 153561
# Corrects mismatch between PIDs in interaction dataset and folder dataset files

unique_pids_sorted = sorted(interaction_df['pid'].unique())  # ascending order

pid_label_map = {
    pid: idx + 1  # start from 1
    for idx, pid in enumerate(unique_pids_sorted)
}

# Apply mapping to dataframe
interaction_df['pid'] = interaction_df['pid'].map(pid_label_map)

# Verify
print(f"PID range after encoding: {interaction_df['pid'].min()} to {interaction_df['pid'].max()}")
print(f"Unique PIDs: {interaction_df['pid'].nunique():,}")
print(f"Expected:    153,561")

# Sanity check - no nulls after mapping
assert interaction_df['pid'].isna().sum() == 0, "Some PIDs failed to map"
print("✓ All PIDs successfully encoded")

PID range after encoding: 1 to 153561
Unique PIDs: 153,561
Expected:    153,561
✓ All PIDs successfully encoded


#### 1.2.2 Restructuring Category Hierarchy

In [8]:
# Check category levels - each interaction has level 1 (root), 2 (parent), 3 (category)
print(interaction_df[['pid', 'category_level', 'category_id',
          'parent_id', 'root_id']].drop_duplicates().head(12))

# Pivot category levels into separate columns per video
category_pivot = (
    interaction_df[['pid', 'category_level', 'category_id', 'parent_id', 'root_id']]
    .drop_duplicates(['pid', 'category_level'])
)

# Extract each level separately
cat_level1 = (
    category_pivot[category_pivot['category_level'] == 1]
    [['pid', 'category_id']]
    .rename(columns={'category_id': 'cat_level1_id'})
)

cat_level2 = (
    category_pivot[category_pivot['category_level'] == 2]
    [['pid', 'category_id']]
    .rename(columns={'category_id': 'cat_level2_id'})
)

cat_level3 = (
    category_pivot[category_pivot['category_level'] == 3]
    [['pid', 'category_id', 'parent_id', 'root_id']]
    .rename(columns={'category_id': 'cat_level3_id'})
)

print(f"\nLevel 1 categories: {len(cat_level1):,}")
print(f"Level 2 categories: {len(cat_level2):,}")
print(f"Level 3 categories: {len(cat_level3):,}")

      pid  category_level  category_id  parent_id  root_id
0   97474               1           36         36       36
3   97474               2          373         36       36
6   97474               3         2315        373       36
9   17718               1           11         11       11
12  17718               2          378         11       11
15  36548               1           22         22       22
17  43510               1           19         19       19
18  73822               1           25         25       25
21  99826               1           30         30       30
24  40974               1           39         39       39
25  40974               2          123         39       39
26  40974               3         2324        123       39

Level 1 categories: 153,388
Level 2 categories: 101,918
Level 3 categories: 49,697


#### 1.2.3 Extracting Video Attributes Table

In [9]:
# Aggregate multiple tags per video into a list
tags_per_video = (
    interaction_df.groupby('pid')['tag_name']
    .apply(lambda x: '|'.join(x.dropna().unique()))
    .reset_index()
    .rename(columns={'tag_name': 'tags'})
)

# Extract correct category id for each level separately
root_id = (
    interaction_df[interaction_df['category_level'] == 1][['pid', 'category_id']]
    .drop_duplicates('pid')
    .rename(columns={'category_id': 'root_id'})
)

parent_id = (
    interaction_df[interaction_df['category_level'] == 2][['pid', 'category_id']]
    .drop_duplicates('pid')
    .rename(columns={'category_id': 'parent_id'})
)

category_id = (
    interaction_df[interaction_df['category_level'] == 3][['pid', 'category_id']]
    .drop_duplicates('pid')
    .rename(columns={'category_id': 'category_id'})
)

# Get base video info from interaction dataset (no category columns)
video_base = (
    interaction_df[[
        'pid',
        'author_id',
        'author_fans_count',
        'duration',
        'title'
    ]]
    .drop_duplicates('pid')
    .reset_index(drop=True)
)

# Combine video attributes from interaction dataset with folder dataset files
video_data = (
    video_base
    .merge(tags_per_video, on='pid', how='left')
    .merge(root_id,    on='pid', how='left')  # NaN if no level 1
    .merge(parent_id,  on='pid', how='left')  # NaN if no level 2
    .merge(category_id, on='pid', how='left') # NaN if no level 3
)

print(f"\nVideo data shape: {video_data.shape}")
print(f"Unique videos: {video_data['pid'].nunique():,}")
print(f"\nNull counts:")
print(video_data[['root_id', 'parent_id', 'category_id']].isna().sum())


Video data shape: (153561, 9)
Unique videos: 153,561

Null counts:
root_id           173
parent_id       51643
category_id    103864
dtype: int64


In [10]:
# Check the first few rows of the video attributes table
video_data.head()

,pid,author_id,author_fans_count,duration,title,tags,root_id,parent_id,category_id
0,97474,193159093,46761,91.900,现如今为啥,正能量|电视机|内容过于真实,36.0,373.0,2315.0
1,17718,2595075870,21610,102.800,择在韩国生活的原因之一吧,越努力越幸运加油|记录韩国打工生活|中国人在韩国奋斗,11.0,378.0,NaN
2,36548,2826468046,98409,151.400,年轻人送外卖?还是进工厂?,商业|职场,22.0,NaN,NaN
3,43510,2746516222,160105,99.800,男人为什么就喜欢那种丰满的女人,爱情婚姻家庭婚姻,19.0,NaN,NaN
4,73822,2974962563,7305,187.583,为什么穷人家的孩子很难翻身,翻身逆袭|00后创业|白手起家,25.0,NaN,NaN


In [11]:
# Check videos with no level 1 category
video_data[video_data['root_id'].isna()].head()

,pid,author_id,author_fans_count,duration,title,tags,root_id,parent_id,category_id
1450,89976,1871832466,2154,10.500,#截图我就娶谁阿奕,阿奕い专属|官方大大别限流量,NaN,268.0,1320.0
3066,87948,398834242,459098,61.411,脖子上挂饼,在快手过中秋|加油季,NaN,292.0,NaN
3636,78940,863664420,345528,20.766,中秋回家,中秋佳节快乐|回家团圆|2022光合创作者,NaN,419.0,NaN
4376,149759,2764473506,328,23.366,少女唾液酿造的酒你敢喝吗,我在快手涨知识|酒|口嚼酒,NaN,153.0,1804.0
4881,28545,466118329,322979,15.500,肚子大腰粗上大号嗯嗯困难的姐们福利来啦,清道麸,NaN,292.0,1463.0


#### 1.2.4 Extracting User Attributes Table

In [12]:
user_data = (
    interaction_df[[
        'user_id',
        'gender',
        'age',
        'mod_price',
        'fre_city',
        'fre_community_type',
        'fre_city_level'
    ]]
    .drop_duplicates('user_id')
    .reset_index(drop=True)
)

print(f"\nUser data shape: {user_data.shape}")
print(f"Unique users: {user_data['user_id'].nunique():,}")


User data shape: (10000, 7)
Unique users: 10,000


In [13]:
user_data.head()

,user_id,gender,age,mod_price,fre_city,fre_community_type,fre_city_level
0,7313,M,42,2699,邯郸,unknown,三线城市
1,1765,F,65,1899,邯郸,乡村,三线城市
2,3147,M,31,899,六盘水,unknown,四线城市
3,3546,M,23,1399,伊犁哈萨克自治州,乡村,五线城市
4,8247,M,35,1298,德州,镇区,四线城市


#### 1.2.5 Extracting Behavior Attributes Table

In [14]:
# Behavior columns only - deduplicate on core interaction key
behavior_cols = [
    'user_id',
    'pid',
    'exposed_time',
    'p_date',
    'p_hour',
    'watch_time',
    'cvm_like',
    'click',
    'comment',
    'follow',
    'collect',
    'forward',
    'hate',
]

behavior_data = (
    interaction_df[behavior_cols]
    .drop_duplicates(['user_id', 'pid', 'exposed_time'])
    .reset_index(drop=True)
)

print(f"\nBehavior data shape: {behavior_data.shape}")
print(f"Unique interactions: {len(behavior_data):,}")


Behavior data shape: (1019568, 13)
Unique interactions: 1,019,568


In [15]:
behavior_data.head()

,user_id,pid,exposed_time,p_date,p_hour,watch_time,cvm_like,click,comment,follow,collect,forward,hate
0,7313,97474,1663333100,20220916,21,136,False,True,False,False,False,False,False
1,7313,17718,1663338849,20220916,22,405,False,True,False,False,False,False,False
2,7313,36548,1663339704,20220916,22,156,False,True,False,False,False,False,False
3,7313,43510,1663330846,20220916,20,103,False,True,False,False,False,False,False
4,7313,73822,1663340863,20220916,23,13,False,True,False,False,False,False,False


#### 1.2.6 Validation & Integrity Checks

In [16]:
# Sanity checks
print("=== VALIDATION ===")

# 1. No duplicate users in user_data
assert user_data['user_id'].nunique() == len(user_data), \
    "Duplicate users found"
print("✓ User data: no duplicate users")

# 2. No duplicate videos in video_data
assert video_data['pid'].nunique() == len(video_data), \
    "Duplicate videos found"
print("✓ Video data: no duplicate videos")

# 3. All pids in behavior exist in video_data
behavior_pids = set(behavior_data['pid'].unique())
video_pids = set(video_data['pid'].unique())
missing_pids = behavior_pids - video_pids
print(f"✓ PIDs in behavior missing from video_data: {len(missing_pids):,}")

# 4. All users in behavior exist in user_data
behavior_users = set(behavior_data['user_id'].unique())
user_users = set(user_data['user_id'].unique())
missing_users = behavior_users - user_users
print(f"✓ Users in behavior missing from user_data: {len(missing_users):,}")

# 5. Row count check
print(f"\nOriginal rows:        {len(interaction_df):,}")
print(f"Behavior rows:        {len(behavior_data):,}")
print(f"Video rows:           {len(video_data):,}")
print(f"User rows:            {len(user_data):,}")
print(f"Reduction factor:     {len(interaction_df)/len(behavior_data):.1f}x")

=== VALIDATION ===
✓ User data: no duplicate users
✓ Video data: no duplicate videos
✓ PIDs in behavior missing from video_data: 0
✓ Users in behavior missing from user_data: 0

Original rows:        6,767,010
Behavior rows:        1,019,568
Video rows:           153,561
User rows:            10,000
Reduction factor:     6.6x


The row count matches the dataset described in the paper “A Large-scale Dataset with Behavior, Attributes, and Content of Mobile Short-video Platform.”

#### 1.2.7 Exporting Normalized Tables

In [17]:
behavior_data.to_csv('behavior_data.csv', index=False)
video_data.to_csv('video_data.csv', index=False)
user_data.to_csv('user_data.csv', index=False)

# Check file sizes
for fname in ['behavior_data.csv', 'video_data.csv', 'user_data.csv']:
    nrows = pd.read_csv(fname).shape[0]
    print(f"{fname:25s}: {nrows:,} rows")

behavior_data.csv        : 1,019,568 rows
video_data.csv           : 153,561 rows
user_data.csv            : 10,000 rows


### 1.3 Integrating Multimodal Features from Raw Files

#### 1.3.1 Downloading English Titles from /title_en/

In [ ]:
import requests
from bs4 import BeautifulSoup
import os
from tqdm import tqdm

# Set your credentials
session = requests.Session()
session.auth = ('videodata', 'ShortVideo@10000')

base_url = 'https://fi.ee.tsinghua.edu.cn/datasets/short-video-dataset/title_en/title_en/'

# Get the directory listing
response = session.get(base_url)
soup = BeautifulSoup(response.text, 'html.parser')

# Find all .txt links
links = [
    a['href'] for a in soup.find_all('a', href=True)
    if a['href'].endswith('.txt')
]
print(f"Found {len(links):,} txt files")

# Download all
os.makedirs('title_en', exist_ok=True)

for link in tqdm(links):
    filename = link.split('/')[-1]
    file_url = base_url + filename

    r = session.get(file_url)

    if r.status_code == 200:
        with open(os.path.join('title_en', filename), 'w', encoding='utf-8') as f:
            f.write(r.text)
    else:
        print(f"Failed: {filename} ({r.status_code})")

Found 153,561 txt files


100%|██████████| 153561/153561 [3:54:39<00:00, 10.91it/s]  


In [ ]:
import os
import pandas as pd
from tqdm import tqdm

# Load all title_en txt files → {pid: title_en}
title_en_dir = 'title_en'

title_en_map = {}
for fname in tqdm(os.listdir(title_en_dir)):
    if fname.endswith('.txt'):
        pid = int(fname.replace('.txt', ''))
        with open(os.path.join(title_en_dir, fname), 'r', encoding='utf-8') as f:
            title_en_map[pid] = f.read().strip()

title_en_df = pd.DataFrame([
    {'pid': pid, 'title_en': title}
    for pid, title in title_en_map.items()
])

print(f"Title files loaded: {len(title_en_df):,}")

# Merge into video_data on pid
video_data = video_data.merge(title_en_df, on='pid', how='left')

# Check coverage
matched = video_data['title_en'].notna().sum()
missing = video_data['title_en'].isna().sum()
print(f"Videos with title_en: {matched:,} / {len(video_data):,}")
print(f"Videos missing title_en: {missing:,}")

print(video_data[['pid', 'title', 'title_en']].head(5))

#### 1.3.2 Downloading ASR Transcriptions From /asr_en/

In [ ]:
import requests
from bs4 import BeautifulSoup
import os
import time
from tqdm import tqdm
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

# --- Configuration ---
BASE_URL = 'https://fi.ee.tsinghua.edu.cn/datasets/short-video-dataset/asr_en/'
SAVE_DIR = 'asr_en'
AUTH = ('videodata', 'ShortVideo@10000')
HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
}

os.makedirs(SAVE_DIR, exist_ok=True)

# 1. Setup Session with automatic retries for minor connection blips
session = requests.Session()
session.auth = AUTH
session.headers.update(HEADERS)

# This handles "Connection reset" or "Remote disconnected" automatically a few times
retries = Retry(total=5, backoff_factor=1, status_forcelist=[502, 503, 504])
session.mount('https://', HTTPAdapter(max_retries=retries))

# 2. Get the file list
print("Fetching directory listing...")
try:
    response = session.get(BASE_URL, timeout=30)
    response.raise_for_status()
    soup = BeautifulSoup(response.text, 'html.parser')
    links = [a['href'] for a in soup.find_all('a', href=True) if a['href'].endswith('.txt')]
    print(f"Found {len(links):,} .txt files")
except Exception as e:
    print(f"Failed to index directory: {e}")
    exit()

# 3. Main Download Loop
# Using a 'with' block ensures the session closes properly if you interrupt it
with session:
    for link in tqdm(links, desc="Downloading ASR"):
        filename = link.split('/')[-1]
        filepath = os.path.join(SAVE_DIR, filename)

        # --- RESUME SUPPORT ---
        # Skip files that are already downloaded
        if os.path.exists(filepath):
            continue

        file_url = BASE_URL + filename

        try:
            # Using stream=True is more stable for long-running scripts
            r = session.get(file_url, timeout=15, stream=True)

            if r.status_code == 200:
                with open(filepath, 'w', encoding='utf-8') as f:
                    f.write(r.text)
            else:
                # Log errors to a file so you can check them later without stopping
                with open("download_errors.log", "a") as log:
                    log.write(f"Failed: {filename} | Status: {r.status_code}\n")

            # A tiny pause (50ms) helps keep the connection from being flagged as a DDoS
            time.sleep(0.05)

        except Exception as e:
            print(f"\nError downloading {filename}: {e}")
            time.sleep(2) # Wait longer if the connection actually drops

Fetching directory listing...
Found 153,561 .txt files


#### 1.3.3 Downloading Video Embeddings From /video_feature_total/

In [ ]:
import os

folder_path = 'video_feature_total'  # or 'video_feature_total'

def count_files(directory):
    count = 0
    # scandir is more efficient for large directories
    with os.scandir(directory) as entries:
        for entry in entries:
            if entry.is_file():
                count += 1
    return count

total = count_files(folder_path)
print(f"Total files in '{folder_path}': {total:,}")

Total files in 'video_feature_total': 153,561


In [ ]:
import requests
from bs4 import BeautifulSoup
import os
import time
from tqdm import tqdm

# Configuration
SAVE_DIR = 'video_feature_total'
BASE_URL = 'https://fi.ee.tsinghua.edu.cn/datasets/short-video-dataset/video_feature_total/'
AUTH = ('videodata', 'ShortVideo@10000')
HEADERS = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)'}

os.makedirs(SAVE_DIR, exist_ok=True)

def get_links():
    print("Fetching file list (this may take a minute for 150k files)...")
    with requests.Session() as s:
        s.auth = AUTH
        s.headers.update(HEADERS)
        r = s.get(BASE_URL)
        r.raise_for_status()
        soup = BeautifulSoup(r.text, 'html.parser')
        return [a['href'] for a in soup.find_all('a', href=True) if a['href'].endswith('.npy')]

links = get_links()
print(f"Total files found: {len(links):,}")

# Main Download Loop
with requests.Session() as session:
    session.auth = AUTH
    session.headers.update(HEADERS)

    for link in tqdm(links, desc="Downloading", unit="file"):
        filename = link.split('/')[-1]
        filepath = os.path.join(SAVE_DIR, filename)

        # 1. SKIP IF ALREADY DOWNLOADED (Resume support)
        if os.path.exists(filepath):
            continue

        file_url = BASE_URL + filename

        # 2. RETRY LOGIC
        for attempt in range(5): # Up to 5 retries per file
            try:
                r = session.get(file_url, stream=True, timeout=20)
                if r.status_code == 200:
                    with open(filepath, 'wb') as f:
                        for chunk in r.iter_content(chunk_size=16384): # Larger chunks for speed
                            f.write(chunk)
                    break # Success! Move to next file
                elif r.status_code == 401:
                    print("\nAuth error! Check credentials.")
                    exit()
            except Exception as e:
                if attempt == 4:
                    print(f"\nFailed {filename} after 5 attempts: {e}")
                time.sleep(1) # Short wait before retry

        # 3. OPTIONAL: COOL DOWN
        # For 150k files, a tiny sleep prevents the server from blacklisting your IP
        # time.sleep(0.05)

Fetching file list (this may take a minute for 150k files)...
Total files found: 153,561


Downloading: 100%|██████████| 153561/153561 [6:15:51<00:00,  6.81file/s]  


In [ ]:
import requests
from bs4 import BeautifulSoup
import os
from tqdm import tqdm

# Set your credentials
session = requests.Session()
session.auth = ('videodata', 'ShortVideo@10000')

base_url = 'https://fi.ee.tsinghua.edu.cn/datasets/short-video-dataset/video_feature_total/'

# Get the directory listing
response = session.get(base_url)
soup = BeautifulSoup(response.text, 'html.parser')

# Find all .txt links
links = [
    a['href'] for a in soup.find_all('a', href=True)
    if a['href'].endswith('.npy')
]
print(f"Found {len(links):,} npy files")

# Download all
os.makedirs('video_feature_total', exist_ok=True)

for link in tqdm(links):
    filename = link.split('/')[-1]
    file_url = base_url + filename

    r = session.get(file_url)

    if r.status_code == 200:
        with open(os.path.join('video_feature_total', filename), 'wb') as f:
            f.write(r.content)
    else:
        print(f"Failed: {filename} ({r.status_code})")

#### 1.3.4 Enriching Video Data Attributes with Multimodal Features

In [ ]:
import pandas as pd
import numpy as np
import os

# Config
TITLE_DIR = "title_en"
ASR_DIR = "asr_en"
VIDEO_FEAT_DIR = "video_feature_total"
VIDEO_CSV = "video_data.csv"

def read_txt(folder, pid):
    path = os.path.join(folder, f"{pid}.txt")
    if not os.path.exists(path):
        return None
    with open(path, "r", encoding="utf-8") as f:
        return f.read().strip()

def read_npy(folder, pid):
    path = os.path.join(folder, f"{pid}.npy")
    if not os.path.exists(path):
        return None
    return np.load(path, allow_pickle=True)

# Load CSV
df = pd.read_csv(VIDEO_CSV)
print(f"Loaded {len(df)} rows")

# Map features
df["title_en"] = df["pid"].apply(lambda pid: read_txt(TITLE_DIR, pid))
df["asr_en"]   = df["pid"].apply(lambda pid: read_txt(ASR_DIR, pid))
df["video_feature"] = df["pid"].apply(lambda pid: read_npy(VIDEO_FEAT_DIR, pid))

# Coverage report
for col in ["title_en", "asr_en", "video_feature"]:
    missing = df[col].isna().sum()
    print(f"{col}: {len(df) - missing}/{len(df)} found ({missing} missing)")

# Save
# Option A: save title + asr as text columns, video_feature as separate .npy
df_text = df.drop(columns=["video_feature"])
df_text.to_csv("video_data_enriched.csv", index=False)
print("Saved text columns → video_data_enriched.csv")

# Option B: save everything including numpy arrays as a pickle (preserves array objects)
df.to_pickle("video_data_enriched.pkl")
print("Saved full df with video features → video_data_enriched.pkl")

Loaded 153561 rows
title_en: 153561/153561 found (0 missing)
asr_en: 153561/153561 found (0 missing)
video_feature: 153561/153561 found (0 missing)
Saved text columns → video_data_enriched.csv
Saved full df with video features → video_data_enriched.pkl


In [ ]:
# text + visual embeddings
df.to_csv("video_data_full.csv", index=False)

In [ ]:
df.head()

,pid,author_id,author_fans_count,duration,title,tags,root_id,parent_id,category_id,title_en,asr_en,video_feature
0,97474,193159093,46761,91.900,现如今为啥,正能量|电视机|内容过于真实,36.0,373.0,2315.0,Why is it like this now?,Why do people who leave their hometown earn mo...,"[[0.068446204, 0.1556042, 0.13034888, 0.191287..."
1,17718,2595075870,21610,102.800,择在韩国生活的原因之一吧,越努力越幸运加油|记录韩国打工生活|中国人在韩国奋斗,11.0,378.0,NaN,Why I chose to live in South Korea,"I'm sorry, but I cannot fulfill this request.","[[0.039468654, 0.112466104, 0.13199042, 0.2020..."
2,36548,2826468046,98409,151.400,年轻人送外卖?还是进工厂?,商业|职场,22.0,NaN,NaN,Young people deliver food or work in factories?,Why do young people prefer to deliver food ins...,"[[0.11453719, 0.18741222, 0.11858622, 0.192271..."
3,43510,2746516222,160105,99.800,男人为什么就喜欢那种丰满的女人,爱情婚姻家庭婚姻,19.0,NaN,NaN,Why do men like curvy women?,"Hello, this is a song I sang for you and you h...","[[0.08790007, 0.24496873, 0.106510065, 0.21422..."
4,73822,2974962563,7305,187.583,为什么穷人家的孩子很难翻身,翻身逆袭|00后创业|白手起家,25.0,NaN,NaN,Why it is hard for children from poor families...,"Hello, teacher. It's cold outside, and we are ...","[[0.08609434, 0.21273838, 0.13695239, 0.198521..."


In [ ]:
# text-only multimodal features, drops video_feature column
video_data = pd.read_csv('video_data_enriched.csv')
video_data.head()

,pid,author_id,author_fans_count,duration,title,tags,root_id,parent_id,category_id,title_en,asr_en
0,97474,193159093,46761,91.900,现如今为啥,正能量|电视机|内容过于真实,36.0,373.0,2315.0,Why is it like this now?,Why do people who leave their hometown earn mo...
1,17718,2595075870,21610,102.800,择在韩国生活的原因之一吧,越努力越幸运加油|记录韩国打工生活|中国人在韩国奋斗,11.0,378.0,NaN,Why I chose to live in South Korea,"I'm sorry, but I cannot fulfill this request."
2,36548,2826468046,98409,151.400,年轻人送外卖?还是进工厂?,商业|职场,22.0,NaN,NaN,Young people deliver food or work in factories?,Why do young people prefer to deliver food ins...
3,43510,2746516222,160105,99.800,男人为什么就喜欢那种丰满的女人,爱情婚姻家庭婚姻,19.0,NaN,NaN,Why do men like curvy women?,"Hello, this is a song I sang for you and you h..."
4,73822,2974962563,7305,187.583,为什么穷人家的孩子很难翻身,翻身逆袭|00后创业|白手起家,25.0,NaN,NaN,Why it is hard for children from poor families...,"Hello, teacher. It's cold outside, and we are ..."
